# 01 — Base documental e embeddings

Global Solution • Assistente Inteligente de Emergências Veiculares

## Objetivo
Inspecionar os documentos fragmentados, a distribuição de categorias, metadados e o índice de embeddings. **Não reprocessa nem treina modelos**.

In [1]:
from pathlib import Path
import json
RAIZ = Path.cwd()
if not (RAIZ / 'base').exists() and (RAIZ.parent / 'base').exists():
    RAIZ = RAIZ.parent
print('Raiz do projeto:', RAIZ.resolve())


Raiz do projeto: C:\Users\marci\PycharmProjects\assistente-ia-veicular


In [2]:
from collections import Counter
arq = RAIZ / 'base' / 'chunks' / 'chunks.json'
chunks = json.loads(arq.read_text(encoding='utf-8'))
print('Total de chunks:', len(chunks))
print('Categorias e contagens:')
for categoria, qtd in sorted(Counter(c['metadados']['categoria'] for c in chunks).items()):
    print(f'  {categoria}: {qtd}')
print('IDs únicos:', len({c['id'] for c in chunks}) == len(chunks))

Total de chunks: 19
Categorias e contagens:
  bateria: 4
  combustivel: 4
  pneus: 4
  seguranca: 4
  superaquecimento: 3
IDs únicos: True


In [3]:
from collections import defaultdict
docs = defaultdict(list)
for c in chunks:
    docs[c['metadados']['documento']].append(c)
for documento, lista in sorted(docs.items()):
    fontes = sorted({f for c in lista for f in c['metadados'].get('fontes', [])})
    print(f'\n{documento}: {len(lista)} chunks')
    for fonte in fontes: print(' Fonte:', fonte)


bateria_sistema_partida.md: 4 chunks
 Fonte: https://www.detran.ma.gov.br/inicio/paginas/Pagina.xhtml?id=23741

pane_seca.md: 4 chunks
 Fonte: https://www.ba.gov.br/detran/noticias/2021-06/04/condutas-de-seguranca-no-transito-pouco-praticadas
 Fonte: https://www.gov.br/prf/pt-br/noticias/estaduais/mato-grosso/2022/outubro/atendimento-por-pane-seca-reduz-10-na-br-163-mt
 Fonte: https://www.planalto.gov.br/ccivil_03/leis/l9503compilado.htm

seguranca_emergencias.md: 4 chunks
 Fonte: https://www.bibliotecajuridica.sp.gov.br/veiculo-ficou-imovel-na-via-detran-sp-orienta-sobre-o-que-fazer/
 Fonte: https://www.gov.br/prf/pt-br/seguranca-viaria/orientacoes-de-viagem
 Fonte: https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-contran/resolucoes/mbvt20222.pdf

superaquecimento_motor.md: 3 chunks
 Fonte: https://www.ford.com.br/content/dam/Ford/website-assets/latam/br/nameplate/2020/ecosport/pdf/Ecosport-Manual%20do%20Propriet%C3%A1rio-MY20.pdf

troca_pneus.md: 4 chunks
 Fonte: http

In [4]:
indice = json.loads((RAIZ / 'base' / 'chunks' / 'indice_embeddings.json').read_text(encoding='utf-8'))
print('Itens no índice:', len(indice))
print('Índice e chunks com os mesmos IDs:', {x['id'] for x in indice} == {x['id'] for x in chunks})
try:
    import numpy as np
    emb = np.load(RAIZ / 'base' / 'chunks' / 'embeddings.npy')
    print('Formato dos embeddings:', emb.shape)
    print('Sem valores inválidos:', bool(np.isfinite(emb).all()))
    assert emb.shape[0] == len(chunks), 'Quantidade de embeddings diferente dos chunks'
except (ImportError, FileNotFoundError) as erro:
    print('Embeddings não inspecionados:', erro)

Itens no índice: 19
Índice e chunks com os mesmos IDs: True
Formato dos embeddings: (19, 384)
Sem valores inválidos: True


## Interpretação
A base contém cinco temas de emergência veicular. O índice e os vetores devem corresponder, em ordem e quantidade, aos chunks armazenados. As fontes são registradas em metadados; sua presença não comprova automaticamente a licença de reutilização.